# HealthConnect Week 7 – Simple KPI Validation
**Goal:** To check that the numbers on the Power BI dashboard are correct, by calculating them again from the original files.

In [1]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


## Load the files
This reads the original files. It does **not** change them.

In [18]:
import pandas as pd

folder = "."   # "." means the files are in the same folder as this notebook

# The clinical workbook has many sheets. sheet_name=None loads all of them.
clinical = pd.read_excel(folder + "/HC_Clinical_Transactions.xlsx", sheet_name=None)
appointments  = clinical["Appointments"]
diagnoses     = clinical["Diagnoses"]
admissions    = clinical["Admissions"]
lab_tests     = clinical["Laboratory_Tests"]
prescriptions = clinical["Prescriptions"]
referrals     = clinical["Referrals"]

# CSV files
billing     = pd.read_csv(folder + "/HC_Billing.csv")
feedback    = pd.read_csv(folder + "/HC_Patient_Feedback.csv")
departments = pd.read_csv(folder + "/HC_Departments.csv")

# Four staff files joined into one table
doctors       = pd.read_csv(folder + "/HC_Doctor.csv")
nurses        = pd.read_csv(folder + "/HC_Nurse.csv")
radiographers = pd.read_csv(folder + "/HC_Radiographer.csv")
admin_staff   = pd.read_csv(folder + "/HC_Administrative_Staff.csv")
staff = pd.concat([doctors, nurses, radiographers, admin_staff])

# Payments and claims are sheets inside your analysis workbook
payments = pd.read_excel(folder + "/HealthConnect_Analysis.xlsx", sheet_name="Payments")
claims   = pd.read_excel(folder + "/HealthConnect_Analysis.xlsx", sheet_name="Insurance_Claims")

print("Appointments:", len(appointments["AppointmentID"]))
print("Billing rows in file:", len(billing))
print("Claims rows in the file:", len(claims))

Appointments: 5000
Billing rows in file: 60000
Claims rows in the file: 42036


In [12]:
appointments.tail(3)

,AppointmentID,PatientID,DoctorID,FacilityID,DepartmentID,Date,VisitType
4997,APT004998,PAT15057,DO0177,HF018,DEP038,2026-03-07,Follow-up
4998,APT004999,PAT15406,DO0333,HF004,DEP140,2026-02-11,Follow-up
4999,APT005000,PAT08439,DO0082,HF015,DEP122,2026-08-20,Follow-up


## Keep only the rows that belong to our 5,000 appointments
Some tables (like billing) have many more rows than there are appointments. We only test the rows that match one of our appointments.

In [13]:
my_appointments = set(appointments["AppointmentID"])   # the 5,000 appointment IDs

billing       = billing[billing["AppointmentID"].isin(my_appointments)].copy()
payments      = payments[payments["BillingID"].isin(billing["BillingID"])].copy()
claims        = claims[claims["BillingID"].isin(billing["BillingID"])].copy()
admissions    = admissions[admissions["AppointmentID"].isin(my_appointments)].copy()
lab_tests     = lab_tests[lab_tests["AppointmentID"].isin(my_appointments)].copy()
prescriptions = prescriptions[prescriptions["AppointmentID"].isin(my_appointments)].copy()
referrals     = referrals[referrals["AppointmentID"].isin(my_appointments)].copy()
feedback      = feedback[feedback["AppointmentID"].isin(my_appointments)].copy()

print("Billing rows kept:", len(billing))
print("Payments kept:", len(payments))
print("Claims kept:", len(claims))
print("Lab tests kept:", len(lab_tests))
print("Prescriptions kept:", len(prescriptions))

Billing rows kept: 5000
Payments kept: 5000
Claims kept: 3521
Lab tests kept: 2272
Prescriptions kept: 4003


## Step 3: Calculate each KPI
Each line calculates one number. Read the comment on the right to see what it means.

In [17]:
total_appointments = len(appointments)                               # number of rows
total_patients    = appointments["PatientID"].nunique()             # different patients

total_billed     = billing["Amount"].sum()                           # add up all bills
average_bill     = billing["Amount"].mean()                          # average bill
cash_received    = payments["AmountPaid"].sum()                      # add up all payments
collection_rate  = cash_received / total_billed                      # share of billed money received
cash_outstanding = total_billed - cash_received                      # money not yet received

total_claims        = len(claims)
approved_claims     = (claims["Status"] == "Approved").sum()         # count the "Approved" ones
claim_approval_rate = approved_claims / total_claims

total_admissions = len(admissions)
admission_rate   = total_admissions / total_appointments
average_stay     = admissions["LengthOfStayDays"].mean()             # days

total_lab_tests   = len(lab_tests)
abnormal_lab_rate = (lab_tests["Result"] == "Abnormal").mean()       # share that are "Abnormal"
total_prescriptions = len(prescriptions)

total_doctors    = (staff["Role"] == "Doctor").sum()
total_staff      = len(staff)
appts_per_doctor = total_appointments / total_doctors

average_rating = feedback["Rating"].mean()

visits_per_patient = appointments.groupby("PatientID").size()        # visits for each patient
one_time_patients  = (visits_per_patient == 1).sum()
frequent_visitors  = (visits_per_patient >= 2).sum()

print("Total appointments :", total_appointments)
print("Total patients    :", total_patients)
print("Total billed       :", round(total_billed))
print("Cash received      :", round(cash_received))
print("Collection rate    :", round(collection_rate * 100, 2), "%")
print("Claim approval rate:", round(claim_approval_rate * 100, 2), "%")
print("Admission rate     :", round(admission_rate * 100, 2), "%")
print("Average stay (days):", round(average_stay, 2))
print("Lab tests          :", total_lab_tests)
print("Abnormal lab rate  :", round(abnormal_lab_rate * 100, 2), "%")
print("Prescriptions      :", total_prescriptions)
print("Doctors / Staff    :", total_doctors, "/", total_staff)
print("Appts per doctor   :", round(appts_per_doctor, 1))
print("Average rating     :", round(average_rating, 2))
print("One-time patients  :", one_time_patients)
print("Frequent visitors  :", frequent_visitors)

Total appointments : 5000
Total patients    : 4443
Total billed       : 771133439
Cash received      : 694551320
Collection rate    : 90.07 %
Claim approval rate: 50.38 %
Admission rate     : 11.72 %
Average stay (days): 5.66
Lab tests          : 2272
Abnormal lab rate  : 49.74 %
Prescriptions      : 4003
Doctors / Staff    : 350 / 1160
Appts per doctor   : 14.3
Average rating     : 2.93
One-time patients  : 3933
Frequent visitors  : 510


## Check the files fit together
These quick checks make sure the tables connect properly.

In [5]:
print("Every appointment has a bill? ", my_appointments.issubset(set(billing["AppointmentID"])))
print("Every bill has a payment?     ", set(billing["BillingID"]).issubset(set(payments["BillingID"])))
print("Any duplicate appointment IDs?", appointments["AppointmentID"].duplicated().any())
print("Any missing values in Appointments?", appointments.isna().sum().sum() > 0)

Every appointment has a bill?  False
Every bill has a payment?      True
Any duplicate appointment IDs? True
Any missing values in Appointments? True


## Compare with the dashboard figures

In [19]:
tests = [
    # KPI name,                  our value,           dashboard value, allowed difference
    ("Total appointments",       total_appointments,  5000,   0.5),
    ("Total patients",          total_patients,     4443,   0.5),
    ("Total billed",             total_billed,        771133439,   0.5),
    ("Average bill",             average_bill,        154227, 0.5),
    ("Cash received",            cash_received,       694551320,  0.5),
    ("Collection rate",          collection_rate,     0.9007, 0.0001),
    ("Cash outstanding",         cash_outstanding,    76582119,   0.5),
    ("Total claims",             total_claims,        3521,   0.5),
    ("Claim approval rate",      claim_approval_rate, 0.5038, 0.0001),
    ("Admissions",               total_admissions,    586,    0.5),
    ("Admission rate",           admission_rate,      0.1172, 0.0001),
    ("Average stay (days)",      average_stay,        5.66,   0.005),
    ("Lab tests",                total_lab_tests,     2272,   0.5),
    ("Abnormal lab rate",        abnormal_lab_rate,   0.4974, 0.0001),
    ("Prescriptions",            total_prescriptions, 4003,   0.5),
    ("Doctors",                  total_doctors,       350,    0.5),
    ("Total staff",              total_staff,         1160,   0.5),
    ("Appointments per doctor",  appts_per_doctor,    14.3,   0.05),
    ("Average rating",           average_rating,      2.93,   0.005),
    ("One-time patients",        one_time_patients,   3933,   0.5),
    ("Frequent visitors",        frequent_visitors,   510,    0.5),
]

def show(number):
    # Makes numbers easy to read: 771133439.0 -> 771,133,439 and 0.90068 -> 0.9007
    return f"{number:,.4f}".rstrip("0").rstrip(".")

rows = []
for name, our_value, dashboard_value, allowed in tests:
    if dashboard_value is None:
        result = "NOT TESTED"
    elif abs(our_value - dashboard_value) <= allowed:
        result = "PASS"
    else:
        result = "FAIL"
    dashboard_text = "" if dashboard_value is None else show(dashboard_value)
    rows.append([name, show(our_value), dashboard_text, result])

results = pd.DataFrame(rows, columns=["KPI", "Our value", "Dashboard value", "Result"])
print(results["Result"].value_counts())
results

Result
PASS    21
Name: count, dtype: int64


,KPI,Our value,Dashboard value,Result
0,Total appointments,"5,000","5,000",PASS
1,Total patients,"4,443","4,443",PASS
2,Total billed,"771,133,439","771,133,439",PASS
3,Average bill,"154,226.6878","154,227",PASS
4,Cash received,"694,551,320","694,551,320",PASS
5,Collection rate,0.9007,0.9007,PASS
6,Cash outstanding,"76,582,119","76,582,119",PASS
7,Total claims,"3,521","3,521",PASS
8,Claim approval rate,0.5038,0.5038,PASS
9,Admissions,586,586,PASS


## Test the facility filter
On your dashboard, set the facility slicer to **HF011** and read the numbers. Then compare them with the numbers below.

In [20]:
facility = "HF011"

fac_appointments = appointments[appointments["FacilityID"] == facility]
fac_ids          = set(fac_appointments["AppointmentID"])
fac_billing      = billing[billing["AppointmentID"].isin(fac_ids)]
fac_feedback     = feedback[feedback["AppointmentID"].isin(fac_ids)]
fac_staff        = staff[staff["FacilityID"] == facility]

print("Facility:", facility)
print("Appointments :", len(fac_appointments))
print("Total billed :", round(fac_billing["Amount"].sum()))
print("Average rating:", round(fac_feedback["Rating"].mean(), 2))
print("Doctors      :", (fac_staff["Role"] == "Doctor").sum())
print("Nurses       :", (fac_staff["Role"] == "Nurse").sum())

Facility: HF011
Appointments : 273
Total billed : 44206632
Average rating: 3.19
Doctors      : 17
Nurses       : 35


In [21]:
# Do the facility rows add up to the overall total? (This tests the facility table on your dashboard.)
bill_with_facility = billing.merge(appointments[["AppointmentID", "FacilityID"]], on="AppointmentID")
billed_by_facility = bill_with_facility.groupby("FacilityID")["Amount"].sum()

print("Sum of all facilities:", round(billed_by_facility.sum()))
print("Overall total billed :", round(total_billed))
print("Result:", "PASS" if round(billed_by_facility.sum()) == round(total_billed) else "FAIL")

Sum of all facilities: 771133439
Overall total billed : 771133439
Result: PASS


## Do the main findings hold for every group?

In [22]:
# Claim approval rate for each insurance provider
claims["Approved"] = claims["Status"] == "Approved"
approval_by_provider = claims.groupby("Provider")["Approved"].mean()
print((approval_by_provider * 100).round(1))

Provider
MMI        49.6
Private    49.3
RSSB       50.9
Radiant    50.1
Name: Approved, dtype: float64


In [23]:
# Appointments per doctor in each department
appts_with_dept = appointments.merge(departments[["DepartmentID", "DepartmentName"]], on="DepartmentID")
appts_by_dept   = appts_with_dept.groupby("DepartmentName").size()
doctors_by_dept = staff[staff["Role"] == "Doctor"].groupby("Department").size()

workload = (appts_by_dept / doctors_by_dept).sort_values(ascending=False)
print(workload.round(1))

DepartmentName
Cardiology                 15.5
Radiology                  15.0
Obstetrics & Gynecology    14.6
Laboratory                 14.3
Emergency                  14.1
Pediatrics                 14.0
Internal Medicine          14.0
Pharmacy                   14.0
Surgery                    13.9
ICU                        13.3
dtype: float64


## Data-quality checks 

In [24]:
# 1. Does payment status match the bill status?
pay_bill = payments.merge(billing, on="BillingID", suffixes=("_payment", "_bill"))
same_status = (pay_bill["Status_payment"] == pay_bill["Status_bill"]).mean()
print(f"Payment status same as bill status: {same_status:.1%}")

# 2. How much of each bill was actually paid?
paid_share = pay_bill["AmountPaid"] / pay_bill["Amount"]
print(f"Payments equal to the full bill:    {(paid_share == 1).mean():.1%}")
print(f"Payments equal to half the bill:    {(paid_share.round(2) == 0.5).mean():.1%}")

# 3. Two ways to measure "money still owed"
owed_by_status = billing[billing["Status"] != "Paid"]["Amount"].sum() / total_billed
owed_by_cash   = cash_outstanding / total_billed
print(f"Outstanding using bill status: {owed_by_status:.1%}")
print(f"Outstanding using payments:    {owed_by_cash:.1%}")

Payment status same as bill status: 100.0%
Payments equal to the full bill:    80.2%
Payments equal to half the bill:    19.8%
Outstanding using bill status: 803.6%
Outstanding using payments:    9.9%


In [25]:
# 4. Does the "Patients by Diagnosis" chart add up to the real number of patients?
patients_by_diagnosis = (appointments
                         .merge(diagnoses[["AppointmentID", "Diagnosis"]], on="AppointmentID")
                         .groupby("Diagnosis")["PatientID"].nunique())
print("Slices add up to:", patients_by_diagnosis.sum())
print("Real number of patients:", unique_patients)
print("Result:", "PASS" if patients_by_diagnosis.sum() == unique_patients else "FAIL (some patients are counted in more than one slice)")

Slices add up to: 4891
Real number of patients: 4443
Result: FAIL (some patients are counted in more than one slice)


In [26]:
results.to_csv("kpi_results.csv", index=False)
print("Saved kpi_results.csv in the same folder as this notebook.")

Saved kpi_results.csv in the same folder as this notebook.
